In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC

In [ ]:
PARTICIPANT_ID = 3
BASE_PATH = '/kaggle/input/datasets/sealeopard/emg-data'
N_FOLDS = 5
RANDOM_SEED = 200
DROP_POSITION_10 = True
FEATURES = 'combine'
 
DAY_CONFIG = {
    1: {'config_label': '+', 'positions': [2, 4, 5, 6, 8]},
    2: {'config_label': 'x', 'positions': [1, 3, 5, 7, 9]},
}
SESSIONS_PER_DAY = [1, 2]

In [ ]:
sample_rate = 2000
window_size_ms = 128
stride_ms = 50
window_size = int(window_size_ms / 1000 * sample_rate)
stride = int(stride_ms / 1000 * sample_rate)
num_channels = 16

notch_freq = 50.0  
q = 30.0
zc_threshold = 0.0
ssc_threshold = 0.0

In [ ]:
FEATURE_CONFIGS = {
    'hudgin': {
        'per_channel_features': 4,       # mean_val, zc, ssc, wl
        'per_channel_size': 16 * 4,      # 64
        'corr_size': 0,
        'total_size': 64,
        'input_order': 'channel_major', 
    },
    'hudgin_9': {
        'per_channel_features': 9,       # mean_val, zc, ssc, wl, rms, var, std, kurt, skew
        'per_channel_size': 16 * 9,      # 144
        'corr_size': 0,
        'total_size': 144,
        'input_order': 'channel_major',
    },
    'sntdf': {
        'per_channel_features': 7,       # mav_nl, p0, p2, p4, p6, diff1, diff2
        'per_channel_size': 16 * 7,      # 112
        'corr_size': 120,                # triu_indices(16, k=1)
        'total_size': 232,
        'input_order': 'channel_major',  
    },
    'combine': {
        # sntdf [232] + hudgin_9 [144] = 376
        # sntdf part: per_channel [112] + corr [120]
        # hudgin_9 part: per_channel [144] + corr [0]
        'per_channel_features': 7,       # sntdf per-channel part (channel_major)
        'per_channel_size': 16 * 7,      # 112  (sntdf per-channel)
        'corr_size': 120,                # sntdf corr_flat
        'hudgin9_size': 144,             # hudgin_9 block (channel_major, 9 features)
        'total_size': 376,
        'input_order': 'channel_major',
    },
}

In [ ]:
def load_all_trials(participant_id):
    all_rows = []
    for day, day_info in DAY_CONFIG.items():
        for session in SESSIONS_PER_DAY:
            csv_path = os.path.join(
                BASE_PATH, f'participant_{participant_id}',
                f'day_{day}', f'session_{session}', 'trials.csv'
            )
            if not os.path.exists(csv_path):
                print(f'[WARN] Not found: {csv_path} — skipped')
                continue
            df = pd.read_csv(csv_path, index_col=0)
            df['day'] = day
            df['session'] = session
            df['config_label'] = day_info['config_label']
            df['local_row'] = df['row_number'].values  
            all_rows.append(df)
            print(f'  Loaded day {day} session {session}: {len(df)} rows '
                  f'(positions={sorted(df["target_position"].unique())})')
 
    full_df = pd.concat(all_rows, ignore_index=True).reset_index(drop=True)
    full_df['row_idx'] = full_df.index  # dùng để tra cứu lại
    return full_df

def relabel_position5_day2_as_10(full_df):
    df = full_df.copy()
    mask = (df['target_position'] == 5) & (df['day'] == 2)
    df.loc[mask, 'target_position'] = 10
    print(f'\n[Relabel] Relabel {mask.sum()} trials: position 5 (day 2) -> position 10')
    return df

def drop_position(full_df, position_to_drop=10):
    df = full_df[full_df['target_position'] != position_to_drop].copy()
    print(f'[Drop] Drop {len(full_df) - len(df)} trials of position {position_to_drop}')
    return df

def verify_structure(full_df):
    print('\n=== Verify dataset configuration ===')
    counts = full_df.groupby(['target_position', 'grasp']).size().unstack(fill_value=0)
    print(counts)
    print('\n Total trials per position:')
    print(full_df.groupby('target_position').size())
    return counts

def generate_folds_per_position(full_df, n_folds=N_FOLDS, seed=RANDOM_SEED):

    fold_results = {}
    for pos in sorted(full_df['target_position'].unique()):
        pos_df = full_df[full_df['target_position'] == pos].reset_index(drop=True)
        local_trial_id = np.arange(len(pos_df))
        row_idx = pos_df['row_idx'].values
        gestures = pos_df['grasp'].values
 
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        fold_results[pos] = {}
        for fold, (train_idx, test_idx) in enumerate(skf.split(local_trial_id, gestures)):
            fold_results[pos][fold] = {
                'train_row_idx': row_idx[train_idx],
                'test_row_idx': row_idx[test_idx],
            }
    return fold_results

def print_fold_balance_check(fold_results, full_df, fold_num=0):
    gesture_by_row_idx = full_df.set_index('row_idx')['grasp']
    print(f'\n=== Check gesture balance (fold {fold_num}, all position) ===')
    for pos in sorted(fold_results.keys()):
        train_rows = fold_results[pos][fold_num]['train_row_idx']
        test_rows = fold_results[pos][fold_num]['test_row_idx']
        from collections import Counter
        train_ges = Counter(gesture_by_row_idx.loc[train_rows].values)
        test_ges = Counter(gesture_by_row_idx.loc[test_rows].values)
        print(f'  Position {pos}: train={len(train_rows)} {dict(sorted(train_ges.items()))}, '
              f'test={len(test_rows)} {dict(sorted(test_ges.items()))}')

def merge_positions(fold_results, fold_num, positions_to_use=None):
    if positions_to_use is None:
        positions_to_use = sorted(fold_results.keys())
    train_rows, test_rows = [], []
    for pos in positions_to_use:
        train_rows.extend(fold_results[pos][fold_num]['train_row_idx'].tolist())
        test_rows.extend(fold_results[pos][fold_num]['test_row_idx'].tolist())
    return np.array(train_rows), np.array(test_rows)

def signal_normalize_window(window, eps=1e-8):
    return window / np.sqrt(np.mean(window ** 2) + eps)
    
def apply_notch_filter(data, fs=sample_rate, freq=notch_freq, q=q):
    b, a = signal.iirnotch(freq, q, fs)
    filtered = signal.lfilter(b, a, data, axis=1)
    return filtered

def offset_correction(data):
    return data - np.mean(data, axis=1, keepdims=True)

def extract_windows(data):
    windows = []
    for start in range(0, data.shape[1] - window_size + 1, stride):
        window = data[:, start:start + window_size]
        windows.append(window)
    return np.array(windows)

def stack_channel_major(per_channel_feats):
    mat = np.stack(per_channel_feats, axis=1)
    return mat.reshape(-1)  
    
def extract_hudgins_features(window):
    if window.ndim != 2:
        raise ValueError("Window must be a 2D array (channels × samples)")
        
    mean_val = np.mean(np.abs(window), axis=1) 
    zc = np.sum(
        (np.diff(np.sign(window), axis=1) != 0)
        & (np.abs(np.diff(window, axis=1)) >= zc_threshold),
        axis=1,
    )
    
    mid   = window[:, 1:-1]
    left  = window[:, :-2]
    right = window[:, 2:]
    ssc = np.sum(((mid - left) * (mid - right)) > ssc_threshold, axis=1)

    wl = np.sum(np.abs(np.diff(window, axis=1)), axis=1)

    return stack_channel_major([mean_val, zc, ssc, wl])

def extract_hudgins_9_features(window):
    if window.ndim != 2:
        raise ValueError("Window must be a 2D array (channels × samples)")
        
    mean_val = np.mean(np.abs(window), axis=1) 
    zc = np.sum(
        (np.diff(np.sign(window), axis=1) != 0)
        & (np.abs(np.diff(window, axis=1)) >= zc_threshold),
        axis=1,
    )
    
    mid   = window[:, 1:-1]
    left  = window[:, :-2]
    right = window[:, 2:]
    ssc = np.sum(((mid - left) * (mid - right)) > ssc_threshold, axis=1)
    
    wl = np.sum(np.abs(np.diff(window, axis=1)), axis=1)

    rms = np.sqrt(np.mean(window**2, axis=1))
    var = np.var(window, axis=1)
    std = np.std(window, axis=1)
    mean_w = np.mean(window, axis=1, keepdims=True)
    std_w = np.where(std < 1e-8, 1e-8, std)[:, None]
    kurt = np.nan_to_num(np.mean(((window - mean_w) / std_w) ** 4, axis=1) - 3.0, nan=0.0)   
    skew = np.nan_to_num(np.mean(((window - mean_w) / std_w) ** 3, axis=1), nan=0.0)

    return stack_channel_major([mean_val, zc, ssc, wl, rms, var, std, kurt, skew])

def extract_sntdf_features(window, eps=1e-8):
    
    d1 = np.diff(window, axis=1)
    d2 = np.diff(window, n=2, axis=1)
    d3 = np.diff(window, n=3, axis=1) 

    mav = np.log(np.mean(np.abs(window), axis=1) + eps)
    p0 = np.log(np.sum(window**2, axis=1) + eps)
    p2 = np.log(np.sum(d1**2, axis=1) + eps)
    p4 = np.log(np.sum(d2**2, axis=1) + eps)
    p6 = np.log(np.sum(d3**2, axis=1) + eps)

    ac1 = np.log(np.mean(np.abs(d1), axis=1) + eps)
    ac2 = np.log(np.mean(np.abs(d2), axis=1) + eps)

    pc = stack_channel_major([mav, p0, p2, p4, p6, ac1, ac2])  # [112]

    corr = np.corrcoef(window)
    corr = np.nan_to_num(corr, nan=0.0) 
    corr_flat = corr[np.triu_indices(num_channels, k=1)]  # Upper triangle
    
    return np.concatenate([pc, corr_flat])

def process_trial(emg_trial, features='hudgin'):
    filtered = apply_notch_filter(emg_trial)
    corrected = offset_correction(filtered)
    windows = extract_windows(corrected)
    #for w in windows:
    #    print("w.shape: ", w.shape)
    #    break
    feats = []
    for w in windows:
        if features == 'hudgin':
            feats.append(extract_hudgins_features(w))
        elif features == 'hudgin_9':
            feats.append(extract_hudgins_9_features(w))
        elif features == 'sntdf':
            feats.append(extract_sntdf_features(signal_normalize_window(w)))
        elif features == 'combine':
            feats.append(np.concatenate([
                extract_sntdf_features(signal_normalize_window(w)),        
                extract_hudgins_9_features(w),       
            ]))
        else:
            raise ValueError(f"Unsupported features: {features}")
        
    return np.array(feats)

def load_emg_for_row_idx(full_df, row_idx_list, base_path=BASE_PATH, participant_id=PARTICIPANT_ID):
    
    trial_lookup = full_df.set_index('row_idx')
    open_files = {}
    results = {}
    missing = []
    
    for ridx in row_idx_list:
        row = trial_lookup.loc[ridx]
        day, session, local_row = row['day'], row['session'], row['local_row']
        hdf5_path = os.path.join(base_path, f'participant_{participant_id}',
                                  f'day_{day}', f'session_{session}', 'emg_data.hdf5')
        if hdf5_path not in open_files:
            open_files[hdf5_path] = h5py.File(hdf5_path, 'r')
        f = open_files[hdf5_path]
        key = str(local_row)
        if key in f:
            results[ridx] = f[key][:]
        else:
            results[ridx] = None
            missing.append({
                'row_idx': ridx,
                'day': day, 'session': session, 'local_row': local_row,
                'target_position': row['target_position'], 'grasp': row['grasp'],
            })
            
    for fh in open_files.values():
        fh.close()

    if missing:
        print(f'\n[WARN] {len(missing)}/{len(row_idx_list)} trial(s) KHÔNG tìm thấy key '
              f'trong hdf5 — đây là dấu hiệu I/O mất data, không phải hành vi bình thường.')
        
        miss_df = pd.DataFrame(missing)
        print('  Phân bố missing theo (position, gesture):')
        print(miss_df.groupby(['target_position', 'grasp']).size())
        
    return results

def build_window_dataset(full_df, row_idx_list, features='hudgin'):

    emg_dict = load_emg_for_row_idx(full_df, row_idx_list)
    lookup = full_df.set_index('row_idx')
 
    features_list, grasp_labels, pos_labels = [], [], []
    for ridx in row_idx_list:
        emg_trial = emg_dict[ridx]
        if emg_trial is None:
            continue
        feat = process_trial(emg_trial, features=features)
        n_windows = feat.shape[0]
        row = lookup.loc[ridx]
        features_list.append(feat)
        grasp_labels.extend([row['grasp']] * n_windows)
        pos_labels.extend([row['target_position']] * n_windows)
 
    X = np.vstack(features_list)
    y_grasp = np.array(grasp_labels)
    y_pos = np.array(pos_labels)
    return X, y_grasp, y_pos

def normalize_data(X_train, X_test, features='hudgin', num_channels=16, eps=1e-8):
    
    X_train = np.asarray(X_train, dtype=np.float64)
    X_test  = np.asarray(X_test,  dtype=np.float64)
    
    cfg = FEATURE_CONFIGS[features]
    assert X_train.shape[1] == cfg['total_size'], (
        f"[{features}] Expected {cfg['total_size']} cols, got {X_train.shape[1]}"
    )

    def _normalize_per_channel_block(Xtr_block, Xte_block, n_feats):

        Xtr_3d = Xtr_block.reshape(-1, num_channels, n_feats)
        Xte_3d = Xte_block.reshape(-1, num_channels, n_feats)

        temp = Xtr_3d.reshape(-1, n_feats)          # [N*C, F]
        mean = temp.mean(axis=0)                     # [F]
        std  = temp.std(axis=0, ddof=1)              # [F], ddof=1 match torch.std
        std  = np.where(std < eps, eps, std)

        Xtr_norm = (Xtr_3d - mean) / std
        Xte_norm = (Xte_3d - mean) / std

        return (Xtr_norm.reshape(Xtr_block.shape[0], -1),
                Xte_norm.reshape(Xte_block.shape[0], -1))

    def _normalize_corr_block(Xtr_block, Xte_block):
        
        mean = Xtr_block.mean(axis=0)               # [120]
        std  = Xtr_block.std(axis=0, ddof=1)        # [120]
        std  = np.where(std < eps, eps, std)

        Xtr_norm = (Xtr_block - mean) / std
        Xte_norm = (Xte_block - mean) / std
        return Xtr_norm, Xte_norm

    if features in ('hudgin', 'hudgin_9'):
        n_feats = cfg['per_channel_features']
        Xtr_norm, Xte_norm = _normalize_per_channel_block(X_train, X_test, n_feats)

    elif features == 'sntdf':
        pc_size  = cfg['per_channel_size']   # 112
        n_feats  = cfg['per_channel_features']  # 7

        # split
        Xtr_pc,   Xte_pc   = X_train[:, :pc_size],  X_test[:, :pc_size]
        Xtr_corr, Xte_corr = X_train[:, pc_size:],  X_test[:, pc_size:]

        # normalize từng block
        Xtr_pc_n,   Xte_pc_n   = _normalize_per_channel_block(Xtr_pc, Xte_pc, n_feats)
        Xtr_corr_n, Xte_corr_n = _normalize_corr_block(Xtr_corr, Xte_corr)

        Xtr_norm = np.hstack([Xtr_pc_n, Xtr_corr_n])
        Xte_norm = np.hstack([Xte_pc_n, Xte_corr_n])

    elif features == 'combine':
        # layout: [sntdf_pc(112) | corr(120) | hudgin9(144)]
        pc_size   = cfg['per_channel_size']    # 112
        corr_size = cfg['corr_size']           # 120
        h9_size   = cfg['hudgin9_size']        # 144

        Xtr_pc   = X_train[:, :pc_size]
        Xte_pc   = X_test[:,  :pc_size]
        Xtr_corr = X_train[:, pc_size:pc_size + corr_size]
        Xte_corr = X_test[:,  pc_size:pc_size + corr_size]
        Xtr_h9   = X_train[:, pc_size + corr_size:]
        Xte_h9   = X_test[:,  pc_size + corr_size:]

        # normalize each block
        Xtr_pc_n,   Xte_pc_n   = _normalize_per_channel_block(Xtr_pc, Xte_pc, 7)
        Xtr_corr_n, Xte_corr_n = _normalize_corr_block(Xtr_corr, Xte_corr)
        Xtr_h9_n,   Xte_h9_n   = _normalize_per_channel_block(Xtr_h9, Xte_h9, 9)

        Xtr_norm = np.hstack([Xtr_pc_n, Xtr_corr_n, Xtr_h9_n])
        Xte_norm = np.hstack([Xte_pc_n, Xte_corr_n, Xte_h9_n])

    else:
        raise ValueError(f"Unsupported features: {features}")

    return Xtr_norm, Xte_norm

def hmc_classification(X_train, X_test, yg_train, yg_test, yp_train, yp_test): 

    X_train_norm, X_test_norm = normalize_data(X_train, X_test, features=FEATURES, num_channels=num_channels)

    #clf_pos = SVC(kernel='rbf', C=1.0, gamma='scale')
    clf_pos = LinearDiscriminantAnalysis()
    clf_pos.fit(X_train_norm, yp_train)
    pos_pred_test = clf_pos.predict(X_test_norm)
    pos_acc = accuracy_score(yp_test, pos_pred_test)

    pos_pred_train = clf_pos.predict(X_train_norm)
    pos_pred_norm_test = (pos_pred_test - np.min(pos_pred_test)) / (np.max(pos_pred_test) - np.min(pos_pred_test) + 1e-8)
    pos_pred_norm_train = (pos_pred_train - np.min(pos_pred_train)) / (np.max(pos_pred_train) - np.min(pos_pred_train) + 1e-8)
    X_aug_test = np.hstack((X_test_norm, pos_pred_norm_test.reshape(-1, 1)))
    X_aug_train = np.hstack((X_train_norm, pos_pred_norm_train.reshape(-1, 1)))

    grasp_clfs = {}
    for pos in np.unique(yp_train):
        mask = (yp_train == pos)
        y_pos_grasp = yg_train[mask]
        if len(np.unique(y_pos_grasp)) > 1:
            clf_g = LinearDiscriminantAnalysis()
            clf_g.fit(X_aug_train[mask], y_pos_grasp)
            grasp_clfs[pos] = clf_g

    preds_grasp = []
    for i in range(len(X_aug_test)):
        p_pred = pos_pred_test[i]
        x_aug = X_aug_test[i].reshape(1, -1)

        pred_g = grasp_clfs[p_pred].predict(x_aug)[0] 
        preds_grasp.append(pred_g)

    preds_grasp = np.array(preds_grasp)

    multi_label_acc = np.mean((pos_pred_test == yp_test) & (preds_grasp == yg_test))
    soft_acc = accuracy_score(yg_test, preds_grasp)
 
    return pos_acc, multi_label_acc, soft_acc

In [ ]:
if __name__ == '__main__':
 
    print(f'=== Pipeline FIXED for participant {PARTICIPANT_ID} ===\n')
 
    print('--- A1: Load trials.csv ---')
    full_df = load_all_trials(PARTICIPANT_ID)
 
    print('\n--- A2: Relabel + Drop position 10 ---')
    full_df = relabel_position5_day2_as_10(full_df)
    if DROP_POSITION_10:
        full_df = drop_position(full_df, position_to_drop=10)
 
    verify_structure(full_df)
 
    print('\n--- A3: StratifiedKFold by gesture, per position ---')
    fold_results = generate_folds_per_position(full_df, n_folds=N_FOLDS)
    print_fold_balance_check(fold_results, full_df, fold_num=0)

    if not os.path.exists(BASE_PATH):
        print(f'\n[BASE_PATH "{BASE_PATH}" NOT EXIST]. ')

    else:
        fold_metrics = []

        for fold in range(N_FOLDS):
        
            train_rows, test_rows = merge_positions(fold_results, fold_num=fold) 
            overlap = set(train_rows) & set(test_rows)
            print(f'Train trials: {len(train_rows)}, Test trials: {len(test_rows)}, '
                  f'overlap (must=0): {len(overlap)}')

            print('--- Build window dataset (train) ---')
 
            X_train, yg_train, yp_train = build_window_dataset(full_df, train_rows, features=FEATURES)
            print(f'X_train shape: {X_train.shape}')
 
            X_test, yg_test, yp_test = build_window_dataset(full_df, test_rows, features=FEATURES)
            print(f'X_test shape: {X_test.shape}')

            for pos in np.unique(yp_train):
                grasps = np.unique(yg_train[yp_train == pos])
                if len(grasps) <= 1:
                    print(f'  [!] position {pos} only has {grasps} grasp in train set '
                          f'-> will miss clf -> potential KeyError')
            missing_pos = set(np.unique(yp_test)) - set(np.unique(yp_train))
            if missing_pos:
                print(f'  [!] position {missing_pos} available in TEST set but not in TRAIN set')

 
            print('\n--- HMC classification ---')
            pos_acc, multi_label_acc, soft_acc = hmc_classification(
                X_train, X_test, yg_train, yg_test, yp_train, yp_test
            )
            print(f'Fold {fold}: Position Acc={pos_acc:.4f}, '
                  f'Multi-label Acc={multi_label_acc:.4f}, Soft Acc={soft_acc:.4f}')

            fold_metrics.append({
                'fold': fold,
                'pos_acc': pos_acc,
                'multi_label_acc': multi_label_acc,
                'soft_acc': soft_acc,
            })

        metrics_df = pd.DataFrame(fold_metrics)
        #print(f'\n{"="*60}\nKẾT QUẢ TỪNG FOLD\n{"="*60}')
        #print(metrics_df)

        print(f'\n{"="*60}\nMEAN 5-FOLD \n{"="*60}')
        for col in ['pos_acc', 'multi_label_acc', 'soft_acc']:
            mean_v = metrics_df[col].mean()
            #std_v = metrics_df[col].std()
            print(f'  {col}: {mean_v:.4f}')

        out_path = f'hmc_fixed_participant_{PARTICIPANT_ID}.csv'
        metrics_df.to_csv(out_path, index=False)
        print(f'\nSave fold results to: {out_path}')
    
    print('\n=== Done ===')